# EE5180 — Seq2Seq (Sutskever et al., 2014) — WMT'14 En→Fr on a Colab GPU

Produces the two reported Table 1 rows (single **forward** vs single **reversed** LSTM),
scored on **newstest2014 — the paper's own ntst14**. Runs the same `src/seq2seq` code as
the local setup; nothing is reimplemented here.

## Just run the next cell

It is the whole pipeline, and it is **idempotent**: it detects what has already finished and
skips it. Free Colab disconnects idle runtimes — when that happens, **re-run that one cell**.
It re-mounts Drive, reinstalls the two packages, and resumes training from the last completed
epoch rather than starting over.

**Before the first run**
1. **Runtime → Change runtime type → T4 GPU.**
2. Put `EE5180_seq2seq_submission` (the folder *or* the `.zip`) in your Drive root.
3. Keep this tab open while it runs. Colab needs a live browser; it will still disconnect
   sometimes, which is exactly what the re-run handles.

Expect ~30 min for data preparation the first time, then roughly 1–2 h per training arm.

> Absolute BLEU is **not** comparable to the paper's Table 1 — see the README's scale-gap
> statement. What reproduces is the direction and shape of the effects.


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  EE5180 — one-cell runner.  RE-RUN THIS WHOLE CELL AFTER ANY DISCONNECT.
#  Everything already finished is detected and skipped, so re-running is cheap.
# ════════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')          # click 'Connect to Google Drive' when asked

import os, sys, subprocess, glob

DRIVE = '/content/drive/MyDrive'
ZIP   = f'{DRIVE}/EE5180_seq2seq_submission.zip'
FOLDER= f'{DRIVE}/EE5180_seq2seq_submission'
REPO  = '/content/seq2seq-ee5180'

# Code runs from Colab's LOCAL disk, not Drive: Drive's FUSE mount is slow for
# the many small reads an import does, and it strips the executable bit.
# Prefer the .zip (always the freshest thing you uploaded); fall back to the folder.
if os.path.exists(ZIP):
    os.makedirs(REPO, exist_ok=True)
    subprocess.run(['unzip', '-q', '-o', ZIP, '-d', REPO], check=True)
elif os.path.isdir(FOLDER):
    subprocess.run(['cp', '-r', FOLDER, REPO], check=True)
else:
    raise SystemExit(f'Put EE5180_seq2seq_submission.zip (or the folder) in {DRIVE}')

if not os.path.exists(f'{REPO}/scripts/colab_run_all.py'):
    inner = glob.glob(f'{REPO}/*/scripts/colab_run_all.py')   # zip with a top-level dir
    if inner:
        REPO = os.path.dirname(os.path.dirname(inner[0]))
    else:
        raise SystemExit('colab_run_all.py missing — upload the CURRENT zip from the repo.')

os.chdir(REPO); sys.path.insert(0, f'{REPO}/src')
# Corpora and checkpoints live on DRIVE so they survive a disconnect.
os.environ['EE5180_WORK'] = f'{DRIVE}/ee5180-work'
print('repo     :', REPO, '(local disk — fast)')
print('work dir :', os.environ['EE5180_WORK'], '(Drive — persistent)')

# torch is already installed and CUDA-built on Colab — do NOT reinstall it.
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'sacrebleu==2.6.0', 'sacremoses==0.2.0', 'pyyaml', 'tqdm'], check=True)
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — set the T4 runtime!')

# Does every outstanding stage: data → train reversed → train forward → decode → report → archive.
!python scripts/colab_run_all.py


## Check progress at any time

Safe to run whenever — reads state off Drive and changes nothing. Useful right after a
reconnect to see how much survived.

(Run the one-cell runner first in a fresh runtime — these cells assume it has set up `os.chdir`.)


In [ ]:
!python scripts/colab_run_all.py --status


## What the runner does, and how to steer it

| Stage | Work |
|---|---|
| `data` | fetch WMT'14, subsample Europarl, tokenise, build vocab, write `.npz` |
| `train_rev` | single **reversed** LSTM — Table 1 row 4 |
| `train_fwd` | single **forward** LSTM — Table 1 row 3 |
| `decode` | beam 1 / 2 / 12 for both arms, scored two ways, plus plots |
| `report` | regenerate the mid-term report from the results |
| `archive` | copy `results/` and `report/` to Drive so they survive a disconnect |

```bash
python scripts/colab_run_all.py --status              # progress, changes nothing
python scripts/colab_run_all.py --dry-run             # show the plan
python scripts/colab_run_all.py --only decode report  # rerun just these
python scripts/colab_run_all.py --seeds 1 2 3 --ensemble   # stretch: ensemble rows
python scripts/colab_run_all.py --only bench          # GPU throughput check
```

Two things it gets right that cost real time to discover: shell scripts are invoked via
`bash` rather than `./` (Drive's FUSE mount strips the executable bit), and decoding never
overlaps training (that contention crippled an earlier local attempt).


## Bring the results back to your Mac

`archive` already copied them to `MyDrive/ee5180-work/deliverables/`, so you can just pull
that folder down. This cell is only if you'd rather download a single archive.


In [ ]:
import shutil, os
shutil.make_archive('/content/ee5180_results', 'zip',
                    os.path.join(os.environ['EE5180_WORK'], 'deliverables'))
print('wrote /content/ee5180_results.zip',
      os.path.getsize('/content/ee5180_results.zip') // 1024, 'KB')
from google.colab import files
files.download('/content/ee5180_results.zip')


Then on the Mac, from the repo:

```bash
make report    # rebuilds the PDF (needs Chrome, which Colab lacks)
make slides    # rebuilds the .pptx (needs node)
```


---
## Manual step-by-step (only if you want to drive stages by hand)

The one-cell runner above already does all of this. These are kept for transparency.


In [ ]:
!python scripts/colab_run_all.py --only data


In [ ]:
!python scripts/colab_run_all.py --only bench


In [ ]:
!python scripts/colab_run_all.py --only train_rev


In [ ]:
!python scripts/colab_run_all.py --only train_fwd


In [ ]:
!python scripts/colab_run_all.py --only decode
print(open('results/wmt14_small/results.md').read())


In [ ]:
!python scripts/colab_run_all.py --only report archive
